# FreshBasket AI — FMCG Demand Forecasting: Model Training & Evaluation 🛒📊

**Objective:** Build, train, and compare multi-horizon forecasting models for weekly unit sales across 1,782 SKU-market combinations ($54 \text{ stores} \times 33 \text{ product families}$) for a 13-week forward procurement window.

### Models Evaluated:
1. **Naive Baseline ($y_{t+h} = y_t$):** Last observed weekly sales volume.
2. **Seasonal Naive (Lag 52):** Same week sales from the previous calendar year.
3. **LightGBM GBDT (Recursive Multiseries):** Supervised gradient boosted trees with 36 leakage-safe features, recursive multi-step forecasting, and empirical uncertainty intervals (80% & 95%).

## 1. Environment Setup & Data Ingestion

In [ ]:
from pathlib import Path
import sys
import joblib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is accessible
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FEATURES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'weekly_features.csv'
MODEL_PATH = PROJECT_ROOT / 'models' / 'trained' / 'lightgbm_model.joblib'
METADATA_PATH = PROJECT_ROOT / 'models' / 'trained' / 'model_metadata.json'
FORECAST_PATH = PROJECT_ROOT / 'data' / 'forecasts' / 'forecast_13_weeks.csv'

print(f"Project Root: {PROJECT_ROOT}")
print(f"Loading weekly features from: {FEATURES_PATH}")

df_features = pd.read_csv(FEATURES_PATH, low_memory=False)
df_features['week_start_date'] = pd.to_datetime(df_features['week_start_date'])

print(f"Total Weekly Rows: {len(df_features):,}")
print(f"Total Series: {df_features[['store_nbr', 'family']].drop_duplicates().shape[0]}")
print(f"Date Range: {df_features['week_start_date'].min().strftime('%Y-%m-%d')} to {df_features['week_start_date'].max().strftime('%Y-%m-%d')}")

## 2. Leakage-Safe Time-Aware Train / Holdout Split

- **Evaluation Cutoff Origin:** `2017-05-14` (Week 227)
- **13-Week Evaluation Window:** `2017-05-15` to `2017-08-13` (Weeks 228 to 240, 23,166 observations)
- **Production Forecast Origin:** `2017-08-13` (Week 240) -> 13 forward weeks

In [ ]:
CUTOFF_DATE = pd.to_datetime("2017-05-14")
HOLDOUT_END_DATE = pd.to_datetime("2017-08-13")

# Filter training eligibility (exclude pre-opening zero periods)
eligible_mask = df_features["is_training_eligible"] == True
train_mask = (df_features["week_start_date"] <= CUTOFF_DATE) & eligible_mask
val_mask = (df_features["week_start_date"] > CUTOFF_DATE) & (df_features["week_start_date"] <= HOLDOUT_END_DATE)

df_train = df_features[train_mask].copy()
df_val = df_features[val_mask].copy()

print(f"Training set rows (<= 2017-05-14): {len(df_train):,}")
print(f"Holdout validation rows (2017-05-15 to 2017-08-13): {len(df_val):,}")

## 3. Inspect Trained LightGBM Model Artifacts

Let us inspect the serialized LightGBM model saved in `models/trained/lightgbm_model.joblib` and its metadata configuration.

In [ ]:
# Load trained model metadata
with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

print("=== Trained Model Architecture & Metadata ===")
print(f"Model Type: {metadata.get('model_type')}")
print(f"Engine: {metadata.get('engine')}")
print(f"Feature Count: {len(metadata.get('features', []))}")
print(f"Cutoff Date: {metadata.get('cutoff_date')}")
print(f"Hyperparameters:\n{json.dumps(metadata.get('hyperparameters', {}), indent=2)}")

# Load serialized model
model = joblib.load(MODEL_PATH)
print(f"\nSuccessfully loaded serialized model object: {type(model).__name__}")

## 4. Multi-Model Holdout Benchmark (Naive vs Seasonal Naive vs LightGBM)

Compare the 3 models on the 13-week holdout using standard FMCG metrics: **WAPE**, **MAE**, **RMSE**, and **RMSLE**.

In [ ]:
METRICS_FILE = PROJECT_ROOT / 'data' / 'forecasts' / 'model_metrics.csv'
df_metrics = pd.read_csv(METRICS_FILE)

# Display summary table
summary_metrics = df_metrics.groupby('model')[['wape', 'mae', 'rmse', 'rmsle']].mean().reset_index()
summary_metrics = summary_metrics.sort_values('wape')
summary_metrics['wape_pct'] = summary_metrics['wape'].apply(lambda x: f"{x:.2f}%")
summary_metrics['mae'] = summary_metrics['mae'].apply(lambda x: f"{x:,.2f}")
summary_metrics['rmse'] = summary_metrics['rmse'].apply(lambda x: f"{x:,.2f}")
summary_metrics['rmsle'] = summary_metrics['rmsle'].apply(lambda x: f"{x:.4f}")

print("=== 13-Week Out-of-Time Holdout Leaderboard ===")
display(summary_metrics[['model', 'wape_pct', 'mae', 'rmse', 'rmsle']])

## 5. 13-Week Forward Forecasts & Prediction Intervals

Inspect the production forecast file (`data/forecasts/forecast_13_weeks.csv`) covering the 13 future weeks across all 1,782 series.

In [ ]:
df_forecast = pd.read_csv(FORECAST_PATH)
print(f"Total 13-week forward predictions: {len(df_forecast):,} rows")
print(f"Columns: {df_forecast.columns.tolist()}\n")

# Display sample forecast with 80% & 95% uncertainty intervals
sample_cols = [
    'date', 'store_nbr', 'family', 'forecast', 
    'lower_bound_80', 'upper_bound_80', 'lower_bound_95', 'upper_bound_95'
]
display(df_forecast[sample_cols].head(10))

## 6. Visualizing Forecasts with Uncertainty Ribbons

Plot the 13-week predicted trajectory for a high-volume category (`GROCERY I` or `BEVERAGES` at Store 1).

In [ ]:
STORE_ID = 1
FAMILY_NAME = 'GROCERY I'

store_fc = df_forecast[(df_forecast['store_nbr'] == STORE_ID) & (df_forecast['family'] == FAMILY_NAME)].copy()
store_fc['date'] = pd.to_datetime(store_fc['date'])
store_fc = store_fc.sort_values('date')

plt.figure(figsize=(12, 6))
plt.plot(store_fc['date'], store_fc['forecast'], marker='o', color='#2563eb', linewidth=2.5, label='Point Forecast (Units)')
plt.fill_between(store_fc['date'], store_fc['lower_bound_80'], store_fc['upper_bound_80'], color='#93c5fd', alpha=0.5, label='80% Prediction Interval')
plt.fill_between(store_fc['date'], store_fc['lower_bound_95'], store_fc['upper_bound_95'], color='#bfdbfe', alpha=0.3, label='95% Prediction Interval')

plt.title(f"13-Week Demand Forecast with Calibrated Uncertainty\nStore {STORE_ID} — {FAMILY_NAME}", fontsize=14, fontweight='bold')
plt.xlabel("Week Start Date", fontsize=11)
plt.ylabel("Weekly Unit Sales", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

## 7. Feature Importance & Driver Attribution

Inspect the primary demand drivers extracted from the trained model.

In [ ]:
ATTRIBUTION_FILE = PROJECT_ROOT / 'reports' / 'feature_attribution.json'
with open(ATTRIBUTION_FILE, 'r') as f:
    attr_data = json.load(f)

group_contrib = attr_data.get('feature_group_importance', {})
groups = list(group_contrib.keys())
weights = [group_contrib[g]['percentage'] for g in groups]

plt.figure(figsize=(10, 5))
bars = plt.barh(groups[::-1], weights[::-1], color='#4f46e5', edgecolor='#3730a3')
plt.xlabel("Relative Importance (%)", fontsize=11)
plt.title("FMCG Demand Driver Attribution (By Feature Group)", fontsize=14, fontweight='bold')
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.5, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va='center', fontsize=10, fontweight='bold')
plt.xlim(0, max(weights) + 10)
plt.tight_layout()
plt.show()